# WSD: алгоритм Леска — Jaccard и sentence-transformers

## 1. Подготовка

In [1]:
%pip install -q nltk sentence-transformers scikit-learn pandas

import os
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"

import math
import re
import time
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd

import nltk
nltk.pathsec.ALLOW_PROXIED_FETCH = True

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

from nltk import pos_tag
from nltk.corpus import wordnet as wn, stopwords

print("NLTK:", nltk.__version__)
print("WordNet:", wn.get_version())

Note: you may need to restart the kernel to use updated packages.
NLTK: 3.10.3
WordNet: 3.0


In [2]:
def get_corpus_path():
    for path in [Path("corpus_wsd_50k.txt"), Path("data/corpus_wsd_50k.txt")]:
        if path.exists():
            return path

    for zip_path in [Path("corpus_wsd_50k.txt.zip"), Path("data/corpus_wsd_50k.txt.zip")]:
        if zip_path.exists():
            with ZipFile(zip_path) as archive:
                member = next(
                    name for name in archive.namelist()
                    if name.endswith("corpus_wsd_50k.txt")
                )
                out_path = zip_path.parent / "corpus_wsd_50k.txt"
                with archive.open(member) as src, open(out_path, "wb") as dst:
                    dst.write(src.read())
            return out_path

    raise FileNotFoundError("Не найден corpus_wsd_50k.txt или corpus_wsd_50k.txt.zip")


corpus_path = get_corpus_path()

raw = corpus_path.read_text(encoding="utf-8").replace("\r\n", "\n")

corpus_wsd = []
for sent in raw.split("\n\n"):
    rows = [line.split("\t") for line in sent.splitlines()]
    rows = [row for row in rows if len(row) == 3]
    if rows:
        corpus_wsd.append(rows)

print("Файл:", corpus_path)
print("Предложений:", f"{len(corpus_wsd):,}")
print("Токенов:", f"{sum(len(s) for s in corpus_wsd):,}")

Файл: corpus_wsd_50k.txt
Предложений: 49,452
Токенов: 1,279,606


## 2. Примеры для оценки и POS

In [4]:
def penn_to_wordnet_pos(tag):
    if tag.startswith("N"):
        return wn.NOUN
    if tag.startswith("V"):
        return wn.VERB
    if tag.startswith("J"):
        return wn.ADJ
    if tag.startswith("R"):
        return wn.ADV
    return None
    
start = time.perf_counter()

sentence_pos = []
for sent in corpus_wsd:
    words = [surface for _, _, surface in sent]
    tags = pos_tag(words)
    sentence_pos.append([penn_to_wordnet_pos(tag) for _, tag in tags])

print(f"Заняло {time.perf_counter() - start:.1f} сек.")

Заняло 27.8 сек.


In [5]:
@dataclass(frozen=True)
class Instance:
    sent_id: int
    token_id: int
    lemma: str
    word: str
    gold: object
    predicted_pos: str | None
    candidates: tuple


@lru_cache(maxsize=None)
def all_candidates(lemma):
    return tuple(wn.synsets(lemma))


@lru_cache(maxsize=None)
def pos_candidates(lemma, pos):
    return tuple(wn.synsets(lemma, pos=pos)) if pos else ()


instances = []
tagged_total = 0
not_ambiguous = 0
gold_not_in_all = 0

for sent_id, sent in enumerate(corpus_wsd):
    for token_id, (sense_key, lemma, word) in enumerate(sent):
        if not sense_key:
            continue

        tagged_total += 1
        gold = wn.lemma_from_key(sense_key).synset()
        candidates_all = all_candidates(lemma)

        if gold not in candidates_all:
            gold_not_in_all += 1
            continue

        # Только случаи, где реально есть выбор
        if len(candidates_all) < 2:
            not_ambiguous += 1
            continue

        predicted_pos = sentence_pos[sent_id][token_id]
        candidates_pos = pos_candidates(lemma, predicted_pos)

        # Если теггер не дал пригодного набора, возвращаемся ко всем значениям.
        candidates = candidates_pos if candidates_pos else candidates_all

        instances.append(
            Instance(
                sent_id=sent_id,
                token_id=token_id,
                lemma=lemma,
                word=word,
                gold=gold,
                predicted_pos=predicted_pos,
                candidates=candidates,
            )
        )

N = len(instances)

print(f"Слов с sense_key: {tagged_total:,}")
print(f"Однозначных по WordNet исключено: {not_ambiguous:,}")
print(f"Gold не найден среди всех кандидатов: {gold_not_in_all:,}")
print(f"Многозначных примеров для оценки: {N:,}")

Слов с sense_key: 239,913
Однозначных по WordNet исключено: 25,817
Gold не найден среди всех кандидатов: 0
Многозначных примеров для оценки: 214,096


In [6]:
def normalize_gold_pos(pos):
    return wn.ADJ if pos == "s" else pos


known_pos = 0
correct_pos = 0
gold_covered = 0

for ins in instances:
    gold_pos = normalize_gold_pos(ins.gold.pos())

    if ins.predicted_pos is not None:
        known_pos += 1
        correct_pos += ins.predicted_pos == gold_pos

    gold_covered += ins.gold in ins.candidates

print(f"POS определена: {100 * known_pos / N:.2f}%")
print(f"POS accuracy: {100 * correct_pos / N:.2f}%")
print(f"Gold остался среди POS-кандидатов: {100 * gold_covered / N:.2f}%")

POS определена: 97.81%
POS accuracy: 91.41%
Gold остался среди POS-кандидатов: 96.10%


## 3. Baseline

In [7]:
def evaluate(predictions, ids=None, decided=None):
    ids = np.arange(N) if ids is None else np.asarray(ids)
    selected = [instances[i] for i in ids]

    ok = np.array(
        [pred == ins.gold for pred, ins in zip(predictions, selected)],
        dtype=bool,
    )

    result = {
        "accuracy": 100 * ok.mean(),
        "correct": int(ok.sum()),
        "total": len(ok),
    }

    if decided is not None:
        decided = np.asarray(decided, dtype=bool)
        if decided.any():
            result["decided_%"] = 100 * decided.mean()
            result["accuracy_decided"] = 100 * ok[decided].mean()

            mfs_ok = np.array(
                [ins.candidates[0] == ins.gold for ins in selected],
                dtype=bool,
            )
            result["mfs_decided"] = 100 * mfs_ok[decided].mean()
        else:
            result["decided_%"] = 0.0
            result["accuracy_decided"] = np.nan
            result["mfs_decided"] = np.nan

    return result


first_sense_pred = [ins.candidates[0] for ins in instances]

random_expected = 100 * np.mean([
    (1 / len(ins.candidates)) if ins.gold in ins.candidates else 0.0
    for ins in instances
])

mfs_result = evaluate(first_sense_pred)

print(f"Random baseline (expected): {random_expected:.2f}%")
print(f"First-sense baseline:       {mfs_result['accuracy']:.2f}%")

Random baseline (expected): 23.30%
First-sense baseline:       63.57%


## 4. Lesk + Jaccard

In [8]:
STOPWORDS = set(stopwords.words("english"))


@dataclass(frozen=True)
class Prep:
    lower: bool = False
    punct: bool = False
    stop: bool = False
    lemma: bool = False


@lru_cache(maxsize=None)
def lemmatize_word(word):
    return wn.morphy(word) or word


def finish_tokens(tokens, prep, lemmatize_tokens):
    if prep.lower:
        tokens = [t.lower() for t in tokens]
    if prep.stop:
        tokens = [t for t in tokens if t.lower() not in STOPWORDS]
    if prep.lemma and lemmatize_tokens:
        tokens = [lemmatize_word(t) for t in tokens]
    return tokens


def tokenize_definition(text, prep):
    tokens = re.findall(r"[A-Za-z]+", text) if prep.punct else text.split()
    return finish_tokens(tokens, prep, lemmatize_tokens=True)


def normalize_corpus(prep):
    norm = []

    for sent in corpus_wsd:
        row = []

        for _, lemma, word in sent:
            text = lemma if prep.lemma else word
            tokens = re.findall(r"[A-Za-z]+", text) if prep.punct else [text]
            row.append(tuple(finish_tokens(tokens, prep, lemmatize_tokens=False)))

        norm.append(row)

    return norm


def gloss_text(synset, mode):
    parts = [synset.definition()]

    if mode in ("def+ex", "ext"):
        parts.extend(synset.examples())

    if mode == "ext":
        parts.extend(h.definition() for h in synset.hypernyms())

    return " ".join(parts)


def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


_norm_cache = {}
_gloss_cache = {}
_jaccard_run_cache = {}

In [15]:
def lesk_jaccard(prep, window=5, gloss_mode="def"):
    run_key = (prep, window, gloss_mode)

    if run_key in _jaccard_run_cache:
        return _jaccard_run_cache[run_key]

    if prep not in _norm_cache:
        _norm_cache[prep] = normalize_corpus(prep)

    norm = _norm_cache[prep]
    gloss_sets = _gloss_cache.setdefault((prep, gloss_mode), {})

    predictions = []
    decided = []

    for ins in instances:
        sent = norm[ins.sent_id]
        i = ins.token_id

        if window is None:
            lo, hi = 0, len(sent)
        else:
            lo = max(0, i - window)
            hi = min(len(sent), i + window + 1)

        # Целевое слово исключаем из контекста
        context = {
            token
            for pos in range(lo, hi)
            if pos != i
            for token in sent[pos]
        }

        scores = []

        for synset in ins.candidates:
            cache_key = synset

            if cache_key not in gloss_sets:
                text = gloss_text(synset, gloss_mode)
                gloss_sets[cache_key] = set(tokenize_definition(text, prep))

            scores.append(jaccard(context, gloss_sets[cache_key]))

        best_score = max(scores)

        # При равенстве выбираем первое (most-frequent sense WordNet)
        predictions.append(ins.candidates[scores.index(best_score)])
        decided.append(best_score > 0)

    result = (predictions, np.asarray(decided, dtype=bool))
    _jaccard_run_cache[run_key] = result
    return result


def run_jaccard_experiments(configs, verbose=False):
    rows = []

    for name, prep, window, gloss_mode in configs:
        start = time.perf_counter()

        pred, decided = lesk_jaccard(
            prep,
            window,
            gloss_mode,
        )

        metrics = evaluate(
            pred,
            decided=decided,
        )

        rows.append({
            "experiment": name,
            "accuracy": metrics["accuracy"],
            "decided_%": metrics["decided_%"],
            "accuracy_decided": metrics["accuracy_decided"],
            "mfs_decided": metrics["mfs_decided"],
            "time_sec": time.perf_counter() - start,
        })

        if verbose:
            print(
                f"{name:<32} "
                f"accuracy={metrics['accuracy']:.2f}% | "
                f"decided={metrics['decided_%']:.2f}%"
            )

    return pd.DataFrame(rows).set_index("experiment")

### 4.1. Предобработка

In [33]:
# Таблица для результатов
def show_table(df, formats=None):
    styler = (
        df.style
        .hide(axis="index")
        .set_properties(
            subset=[df.columns[0]],
            **{
                "text-align": "left",
                "font-weight": "bold",
            }
        )
    )

    if len(df.columns) > 1:
        styler = styler.set_properties(
            subset=list(df.columns[1:]),
            **{"text-align": "center"}
        )

    styler = styler.set_table_styles([
        {
            "selector": "th",
            "props": [("text-align", "center")]
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
                ("font-weight", "bold"),
            ]
        },
    ])

    if formats:
        styler = styler.format(formats)

    display(styler)

In [27]:
prep_configs = [
    ("raw", Prep()),
    ("+ lower", Prep(lower=True)),
    ("+ punctuation", Prep(lower=True, punct=True)),
    ("+ stopwords", Prep(lower=True, punct=True, stop=True)),
    ("+ lemmatization", Prep(lower=True, punct=True, stop=True, lemma=True)),
]

jaccard_prep_results = run_jaccard_experiments([
    (name, prep, 5, "def")
    for name, prep in prep_configs
])

prep_table = (
    jaccard_prep_results
    .round(2)
    .reset_index()
)

show_table(
    prep_table,
    {
        "accuracy": "{:.2f}",
        "decided_%": "{:.2f}",
        "accuracy_decided": "{:.2f}",
        "mfs_decided": "{:.2f}",
        "time_sec": "{:.2f}",
    }
)

best_prep_name = jaccard_prep_results["accuracy"].idxmax()
best_prep = dict(prep_configs)[best_prep_name]

print("Лучший вариант:", best_prep_name)

experiment,accuracy,decided_%,accuracy_decided,mfs_decided,time_sec
raw,34.30,74.45,22.25,61.57,0.26
+ lower,33.82,75.75,22.36,61.63,0.15
+ punctuation,33.52,76.94,22.65,61.71,0.13
+ stopwords,61.77,6.90,29.91,56.05,0.13
+ lemmatization,60.77,10.13,28.77,56.44,0.17


Лучший вариант: + stopwords


### 4.2. Размер контекста

In [28]:
window_options = [2, 3, 5, 10, 20, None]

jaccard_window_results = run_jaccard_experiments([
    (
        f"window={w}" if w is not None else "full sentence",
        best_prep,
        w,
        "def",
    )
    for w in window_options
])

window_table = (
    jaccard_window_results
    .round(2)
    .reset_index()
)

show_table(
    window_table,
    {
        "accuracy": "{:.2f}",
        "decided_%": "{:.2f}",
        "accuracy_decided": "{:.2f}",
        "mfs_decided": "{:.2f}",
        "time_sec": "{:.2f}",
    }
)

best_window_name = jaccard_window_results["accuracy"].idxmax()

best_window = (
    None
    if best_window_name == "full sentence"
    else int(best_window_name.split("=")[1])
)

print("Лучший контекст:", best_window_name)

experiment,accuracy,decided_%,accuracy_decided,mfs_decided,time_sec
window=2,62.89,3.13,33.39,55.10,0.21
window=3,62.48,4.61,31.70,55.25,0.15
window=5,61.77,6.90,29.91,56.05,0.15
window=10,60.42,10.95,28.03,56.78,0.14
window=20,58.99,15.00,26.87,57.37,0.13
full sentence,58.13,17.42,26.44,57.67,0.14


Лучший контекст: window=2


### 4.3. Текст значения WordNet

In [30]:
gloss_options = {
    "definition": "def",
    "definition + examples": "def+ex",
    "definition + examples + hypernyms": "ext",
}

jaccard_gloss_results = run_jaccard_experiments([
    (name, best_prep, best_window, mode)
    for name, mode in gloss_options.items()
])

gloss_table = (
    jaccard_gloss_results
    .round(2)
    .reset_index()
)

show_table(
    gloss_table,
    {
        "accuracy": "{:.2f}",
        "decided_%": "{:.2f}",
        "accuracy_decided": "{:.2f}",
        "mfs_decided": "{:.2f}",
        "time_sec": "{:.2f}",
    }
)

best_gloss_name = jaccard_gloss_results["accuracy"].idxmax()
best_gloss_mode = gloss_options[best_gloss_name]
print("Лучший текст значения:", best_gloss_name)

experiment,accuracy,decided_%,accuracy_decided,mfs_decided,time_sec
definition,62.89,3.13,33.39,55.10,0.24
definition + examples,62.26,8.14,37.52,53.56,0.15
definition + examples + hypernyms,61.94,9.10,35.67,53.61,0.13


Лучший текст значения: definition


In [34]:
jaccard_pred, jaccard_decided = lesk_jaccard(
    best_prep,
    best_window,
    best_gloss_mode,
)

jaccard_final = evaluate(
    jaccard_pred,
    decided=jaccard_decided,
)

jaccard_final_table = pd.DataFrame([
    {
        "метод": "Jaccard",
        "accuracy": jaccard_final["accuracy"],
        "correct": jaccard_final["correct"],
        "total": jaccard_final["total"],
        "decided_%": jaccard_final["decided_%"],
        "accuracy_decided": jaccard_final["accuracy_decided"],
        "mfs_decided": jaccard_final["mfs_decided"],
    }
])

show_table(
    jaccard_final_table,
    {
        "accuracy": "{:.2f}",
        "correct": "{:.0f}",
        "total": "{:.0f}",
        "decided_%": "{:.2f}",
        "accuracy_decided": "{:.2f}",
        "mfs_decided": "{:.2f}",
    }
)

метод,accuracy,correct,total,decided_%,accuracy_decided,mfs_decided
Jaccard,62.89,134644,214096,3.13,33.39,55.10


In [32]:
jaccard_comparison = pd.DataFrame([
    {
        "метод": "Jaccard + first-sense fallback",
        "accuracy": jaccard_final["accuracy"],
    },
    {
        "метод": "Jaccard + random fallback",
        "accuracy": pure_jaccard_expected,
    },
    {
        "метод": "First-sense baseline",
        "accuracy": mfs_result["accuracy"],
    },
    {
        "метод": "Random baseline",
        "accuracy": random_expected,
    },
])

show_table(
    jaccard_comparison,
    {
        "accuracy": "{:.2f}"
    }
)

метод,accuracy
Jaccard + first-sense fallback,62.89
Jaccard + random fallback,23.79
First-sense baseline,63.57
Random baseline,23.30


Лучше всего Jaccard сработал с окном ±2 слова, удалением стоп-слов и обычными определениями WordNet, accuracy получилась 62.89%. Но при этом реальное пересечение слов между контекстом и определением было только в 3.13% случаев, то есть почти всегда Jaccard сам ничего не мог решить, и алгоритм просто выбирал первое значение WordNet. Это видно и по сравнению с first-sense baseline: он дал 63.57%, то есть даже немного больше. Если в тех случаях, где Jaccard не нашёл пересечения, выбирать значение случайно, accuracy падает до 23.79%, что почти совпадает со случайным baseline 23.30%. Поэтому высокий итоговый результат в основном связан не с самим Jaccard, а с выбором первого значения WordNet.

Увеличение окна и добавление примеров или гиперонимов помогали чаще находить пересечения, но итоговое качество от этого не росло. В целом Jaccard для этой задачи оказался довольно слабым, потому что слова в реальном контексте и в словарном определении часто просто не совпадают.

## 5. Lesk + sentence-transformers

In [35]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

start = time.perf_counter()

model = SentenceTransformer(MODEL_NAME)

print(f"Потрачено {time.perf_counter() - start:.1f} сек.")
print("Устройство:", model.device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Потрачено 5.1 сек.
Устройство: mps:0


In [36]:
def detokenize(text):
    text = text.replace("_", " ")
    text = re.sub(
        r"\s+([,.;:!?%)\]]|'s\b|n't\b|'re\b|'ve\b|'ll\b|'d\b|'m\b)",
        r"\1",
        text,
    )
    return re.sub(r"([(\[])\s+", r"\1", text)


def embedding_context_text(ins, window=None, keep_target=True):
    sent = corpus_wsd[ins.sent_id]

    if window is None:
        lo, hi = 0, len(sent)
    else:
        lo = max(0, ins.token_id - window)
        hi = min(len(sent), ins.token_id + window + 1)

    words = []

    for pos in range(lo, hi):
        if pos == ins.token_id and not keep_target:
            continue
        words.append(sent[pos][2])

    return detokenize(" ".join(words))


def embedding_definition_text(ins, synset, mode):
    gloss = synset.definition()

    if mode == "gloss":
        return gloss

    if mode == "lemma: gloss":
        return f"{ins.lemma.replace('_', ' ')}: {gloss}"

    if mode == "gloss+examples":
        examples = synset.examples()
        return gloss if not examples else gloss + "; " + "; ".join(examples)

    raise ValueError(mode)

### 5.1. Подбор конфигурации на фиксированной подвыборке

In [37]:
TUNE_SAMPLE_SIZE = min(5000, N)
SEED = 42

rng = np.random.default_rng(SEED)
tune_ids = np.sort(
    rng.choice(N, size=TUNE_SAMPLE_SIZE, replace=False)
)

print(f"Подвыборка для настройки: {len(tune_ids):,} примеров")

Подвыборка для настройки: 5,000 примеров


In [38]:
EMBEDDING_CACHE = {}


def encode_missing(texts, batch_size=64):
    missing = list(dict.fromkeys(
        text for text in texts
        if text not in EMBEDDING_CACHE
    ))

    if not missing:
        return

    vectors = model.encode(
        missing,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    EMBEDDING_CACHE.update(zip(missing, vectors))


def lesk_embeddings_sample(ids, ctx_window, def_mode, keep_target=True):
    ids = np.asarray(ids)

    contexts = {}
    definitions = {}

    all_texts = []

    for n in ids:
        ins = instances[n]

        context = embedding_context_text(
            ins,
            window=ctx_window,
            keep_target=keep_target,
        )

        defs = [
            embedding_definition_text(ins, synset, def_mode)
            for synset in ins.candidates
        ]

        contexts[n] = context
        definitions[n] = defs

        all_texts.append(context)
        all_texts.extend(defs)

    encode_missing(all_texts)

    predictions = []

    for n in ids:
        ins = instances[n]

        context_vec = EMBEDDING_CACHE[contexts[n]]
        def_matrix = np.stack([
            EMBEDDING_CACHE[text]
            for text in definitions[n]
        ])

        similarity = def_matrix @ context_vec
        cosine_distance = 1.0 - similarity

        predictions.append(
            ins.candidates[int(np.argmin(cosine_distance))]
        )

    return predictions

In [46]:
embedding_configs = [
    ("window ±5 | gloss",              5,    "gloss",          True),
    ("window ±5 | lemma: gloss",       5,    "lemma: gloss",   True),
    ("window ±5 | gloss+examples",     5,    "gloss+examples", True),
    ("full sentence | gloss",          None, "gloss",          True),
    ("full sentence | lemma: gloss",   None, "lemma: gloss",   True),
    ("full sentence | gloss+examples", None, "gloss+examples", True),
]

embedding_rows = []

for name, window, def_mode, keep_target in embedding_configs:
    start = time.perf_counter()

    pred = lesk_embeddings_sample(
        tune_ids,
        ctx_window=window,
        def_mode=def_mode,
        keep_target=keep_target,
    )

    metrics = evaluate(pred, ids=tune_ids)

    embedding_rows.append({
        "experiment": name,
        "accuracy": metrics["accuracy"],
        "time_sec": time.perf_counter() - start,
        "window": window,
        "def_mode": def_mode,
        "keep_target": keep_target,
    })

embedding_results = pd.DataFrame(embedding_rows)

embedding_table = embedding_results[
    ["experiment", "accuracy", "time_sec"]
].round(2)

show_table(
    embedding_table,
    {
        "accuracy": "{:.2f}",
        "time_sec": "{:.2f}",
    }
)

best_embedding_idx = embedding_results["accuracy"].idxmax()
best_embedding_row = embedding_results.loc[best_embedding_idx]

print(
    f"Лучший результат: {best_embedding_row['experiment']} "
    f"с accuracy {best_embedding_row['accuracy']:.2f}%."
)

experiment,accuracy,time_sec
window ±5 | gloss,37.02,0.34
window ±5 | lemma: gloss,33.84,0.19
window ±5 | gloss+examples,40.06,0.20
full sentence | gloss,35.92,0.18
full sentence | lemma: gloss,32.42,0.17
full sentence | gloss+examples,38.28,0.19


Лучший результат: window ±5 | gloss+examples с accuracy 40.06%.


### 5.2. Финальный cosine на всех многозначных примерах

In [47]:
def lesk_embeddings_full(
    window,
    def_mode,
    keep_target=True,
    batch_size=128,
    report_every=100,
):
    definition_texts = {
        embedding_definition_text(ins, synset, def_mode)
        for ins in instances
        for synset in ins.candidates
    }

    print(f"Кодируем недостающие определения: {len(definition_texts):,}")
    encode_missing(definition_texts, batch_size=64)

    predictions = []
    total_batches = math.ceil(N / batch_size)

    print(f"Финальная cosine-оценка: {N:,} примеров")
    start_time = time.perf_counter()

    for batch_num, start_idx in enumerate(
        range(0, N, batch_size),
        start=1,
    ):
        batch = instances[start_idx:start_idx + batch_size]

        contexts = [
            embedding_context_text(
                ins,
                window=window,
                keep_target=keep_target,
            )
            for ins in batch
        ]

        context_vectors = model.encode(
            contexts,
            batch_size=batch_size,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

        for ins, context_vec in zip(batch, context_vectors):
            def_texts = [
                embedding_definition_text(ins, synset, def_mode)
                for synset in ins.candidates
            ]

            def_matrix = np.stack([
                EMBEDDING_CACHE[text]
                for text in def_texts
            ])

            similarity = def_matrix @ context_vec
            cosine_distance = 1.0 - similarity

            predictions.append(
                ins.candidates[int(np.argmin(cosine_distance))]
            )

        if (
            batch_num == 1
            or batch_num % report_every == 0
            or batch_num == total_batches
        ):
            processed = min(start_idx + batch_size, N)
            elapsed = time.perf_counter() - start_time

            speed = processed / elapsed
            eta = (N - processed) / speed if speed else 0

            current = evaluate(
                predictions,
                ids=np.arange(processed),
            )["accuracy"]

            print(
                f"{processed:>7,}/{N:,} | "
                f"{100 * processed / N:6.2f}% | "
                f"accuracy={current:5.2f}% | "
                f"прошло={elapsed:7.1f} сек. | "
                f"осталось≈{eta:7.1f} сек.",
                flush=True,
            )

    return predictions


cosine_pred = lesk_embeddings_full(
    window=best_embedding_window,
    def_mode=best_embedding_def_mode,
    keep_target=best_embedding_keep_target,
)

cosine_final = evaluate(cosine_pred)

cosine_final_table = pd.DataFrame([
    {
        "метод": "Lesk + cosine",
        "accuracy": cosine_final["accuracy"],
        "correct": cosine_final["correct"],
        "total": cosine_final["total"],
    }
])

show_table(
    cosine_final_table,
    {
        "accuracy": "{:.2f}",
        "correct": "{:.0f}",
        "total": "{:.0f}",
    }
)

Кодируем недостающие определения: 33,741


Batches:   0%|          | 0/365 [00:00<?, ?it/s]

Финальная cosine-оценка: 214,096 примеров
    128/214,096 |   0.06% | accuracy=38.28% | прошло=    1.0 сек. | осталось≈ 1682.4 сек.
 12,800/214,096 |   5.98% | accuracy=43.76% | прошло=   88.2 сек. | осталось≈ 1387.5 сек.
 25,600/214,096 |  11.96% | accuracy=42.68% | прошло=  181.6 сек. | осталось≈ 1337.4 сек.
 38,400/214,096 |  17.94% | accuracy=42.48% | прошло=  304.5 сек. | осталось≈ 1393.3 сек.
 51,200/214,096 |  23.91% | accuracy=43.13% | прошло=  412.4 сек. | осталось≈ 1312.0 сек.
 64,000/214,096 |  29.89% | accuracy=43.19% | прошло=  642.4 сек. | осталось≈ 1506.7 сек.
 76,800/214,096 |  35.87% | accuracy=43.29% | прошло=  750.3 сек. | осталось≈ 1341.4 сек.
 89,600/214,096 |  41.85% | accuracy=43.22% | прошло=  896.3 сек. | осталось≈ 1245.3 сек.
102,400/214,096 |  47.83% | accuracy=43.17% | прошло=  991.0 сек. | осталось≈ 1081.0 сек.
115,200/214,096 |  53.81% | accuracy=43.23% | прошло= 1116.9 сек. | осталось≈  958.8 сек.
128,000/214,096 |  59.79% | accuracy=43.19% | прошло= 1268

метод,accuracy,correct,total
Lesk + cosine,40.11,85869,214096


## 6. Выводы

In [48]:
summary = pd.DataFrame([
    {
        "метод": "Random baseline",
        "accuracy": random_expected,
    },
    {
        "метод": "First-sense baseline",
        "accuracy": mfs_result["accuracy"],
    },
    {
        "метод": "Lesk + Jaccard",
        "accuracy": jaccard_final["accuracy"],
    },
    {
        "метод": "Lesk + sentence-transformers / cosine",
        "accuracy": cosine_final["accuracy"],
    },
])

show_table(
    summary,
    {
        "accuracy": "{:.2f}%",
    }
)

print(
    f"\nЛучший Jaccard: "
    f"окно {'всё предложение' if best_window is None else f'±{best_window}'}, "
    f"определение — {best_gloss_name}, "
    f"ненулевое пересечение — {jaccard_final['decided_%']:.2f}%."
)

print(
    f"Лучший cosine: {best_embedding_name}."
)

метод,accuracy
Random baseline,23.30%
First-sense baseline,63.57%
Lesk + Jaccard,62.89%
Lesk + sentence-transformers / cosine,40.11%



Лучший Jaccard: окно ±2, определение — definition, ненулевое пересечение — 3.13%.
Лучший cosine: window ±5 | gloss+examples.


Для cosine лучше всего сработало окно ±5 слов и представление значения через определение WordNet вместе с примерами. На тестовой выборке эта конфигурация дала accuracy 40.06%, а на полном наборе - 40.11%, то есть результат практически не изменился. Это показывает, что выбранная тестовая выборка достаточно хорошо отражала полный датасет. Добавление примеров к определениям улучшило результат, а использование всего предложения вместо небольшого окна, наоборот, немного ухудшило качество. Добавление леммы перед определением тоже не помогло. Получается, что для определения значения слова полезнее локальный контекст рядом с ним, а лишняя информация из всего предложения может добавлять шум.

Cosine заметно лучше случайного baseline: 40.11% / 23.30%. Значит, эмбеддинги действительно улавливают некоторую смысловую связь между контекстом и определениями WordNet. Но результат всё равно намного ниже first-sense baseline — 63.57%. Возможно, что в корпусе первое значение WordNet часто совпадает с правильным, поэтому простой выбор наиболее частого значения будет сильнее семантического сравнения.

Jaccard показал почти такой же высокий результат, как first-sense baseline (62.89%), но этот результат в основном получился из-за fallback на первое значение WordNet. Сам Jaccard нашёл ненулевое пересечение только в 3.13% случаев, а при случайном выборе значения в остальных случаях accuracy была всего 23.79%. 

В итоге cosine для нашей задачи - более содержательный метод: он использует семантическое сходство и превосходит случайный выбор, хотя и уступает сильному first-sense baseline. Jaccard оказался слабым, потому что словарные определения и реальные контексты редко содержат одинаковые слова